# s7 · Part-based RePaint — color inpainting on frozen-PoinTr completions

**Method by Pelin Kılıç** (`RePaint_part.ipynb` / `RePaint_part_reel.ipynb`, shared Drive, 2026-07-31),
after [RePaint, Lugmayr et al., CVPR 2022](https://arxiv.org/abs/2201.09865). This notebook is the
**productionized replication at scale** (ADR-0010): same method, same ΔE protocol, formal experiment
discipline — full test split, multi-seed, the promised 25/50/75% occlusion sweep, config-signed
checkpoints on HF, resumable per-row eval, W&B curves. Core logic lives in tested modules:
`tools/pc_repaint.py` (DDPM + RePaint + eval rows) and `tools/pc_partlabels.py` (labeled real-texture
GT built from **our own** HF bundles + ShapeNet-Part labels — no dependency on anyone's Drive).

**The idea.** Geometry is completed by a **frozen pretrained PoinTr** (works; untouched). The open
question is how the filled-in points are **colored**. The two rule baselines (NN-copy, part-mean) are
deterministic and flat — and have no anchor on a fully occluded part. RePaint replaces the rule: an
**unconditional DDPM over the per-point RGB field** (mask never seen in training), conditioned only
at **sampling time** by re-injecting known colors at the right noise level, with time-travel
resampling. One trained model handles every occlusion ratio.

**Run:** Runtime ▸ GPU, then Run all. Stage is picked by `DATA_SOURCE` in T0:
`"auto"` = synthetic palette colors (the zip-free smoke gate — reproduce her Stage-0 ordering before
any real compute), `"labeled_s3"` = real texture (the formal experiment). Everything downloads or
builds itself; resume after a disconnect = just Run all again (done work is skipped from HF).


In [ ]:
# ============================================================
# Cell T0 — Config  (RUN ME FIRST; the only cell you edit)
# ------------------------------------------------------------
# One flat config. The eval is the formal experiment: full test split, multi-seed,
# occlusion sweep {25,50,75}% at eval time (one model, no retraining — RePaint's point).
# Checkpoints are config-signed (tools/pc_repaint): change anything here and stale
# checkpoints are REJECTED instead of silently poisoning the tables.
# ============================================================

# --- experiment stage ---
DATA_SOURCE = "labeled_s3"        # "auto" (synthetic palette; smoke gate) | "labeled_s3" (real texture)
CATEGORY, SYNSET = "Airplane", "02691156"
N_PTS      = 2048                 # color-cloud resolution (DDPM + part-seg operate here)
N_MODELS   = 0                    # 0 = all available (auto: len(AUTO_SPLITS) files; labeled: bundles∩labels)
TEST_FRAC  = 0.25                 # DDPM + PointNet train ONLY on the train split
CNOISE     = 0.03                 # synthetic color noise (auto stage only)
AUTO_SPLITS = ["train0", "test1"] # her Stage-0 files -> ~150 airplanes (parity with her run)

# --- diffusion / RePaint (her architecture, unchanged) ---
T_STEPS, WIDTH, KNN, N_BLOCKS = 200, 128, 16, 3
DDPM_EPOCHS, BATCH, LR        = 300, 8, 2e-4
SEG_EPOCHS                    = 60
JUMP_LEN, JUMP_N, ADAPTIVE_JUMPS = 10, 3, True

# --- the formal eval ---
EVAL_CROPS      = (0.25, 0.50, 0.75)
HEADLINE_CROP   = 0.50            # her tables' crop; gets the full seed budget
N_SEEDS_HEADLINE, N_SEEDS_OFFCROP, N_SEEDS_E2 = 5, 3, 3
EVAL_N          = 0               # 0 = the whole test split

# --- frozen PoinTr (her fork, pinned into OUR GitHub account — no external dependency) ---
PINNED_FORK   = "https://github.com/efeyenice/Pelin_Efe_PoinTr.git"
FALLBACK_FORK = "https://github.com/eylulpelinkilic/Pelin_Efe_PoinTr.git"
PIN_COMMIT    = "3f98676aeaabd9c1150dd04cd0a51f504949ab9c"
POINTR_GDOWN_ID = "1WzERLlbSwzGOBybzkjBrApwyVMTG00CJ"   # fallback only; HF mirror is primary
POINTR_IN, USE_DENSE_FOR_POINTR = 1024, True            # refined automatically by the T4 scan

# --- storage + tracking (ADR-0008: HF-native, Drive retired) ---
HF_DATA_REPO   = "efeyenice/pc-completion-data"
H5_MIRROR      = "data_labeled/shapenetpart_h5"     # ShapeNet-Part h5 benchmark, mirrored on first touch
LABELED_PATH   = "data_labeled/" + SYNSET           # our fused labeled real-texture GT
POINTR_CKPT_HF = "ckpts/PoinTr_ShapeNet55.pth"      # 400MB PoinTr weights, mirrored on first touch
RUNS_PATH      = "runs/repaint"                     # ckpts + eval_rows.csv + panels live here on HF
LOCAL_RUNS, LOCAL_LABELED, LOCAL_H5 = "/content/repaint_runs", "/content/labeled_gt", "/content/snpart_h5"
REPO_URL       = "https://github.com/efeyenice/point-cloud-completion-research.git"
HF_SYNC_ON, WANDB_ON = True, True
WANDB_ENTITY  = "efeyenice004-sabanc-university-nanotechnology-research-a"
WANDB_PROJECT = "pc-completion"

RUN_TAG = f"repaint_{DATA_SOURCE}_{CATEGORY.lower()}"
print(f"stage={DATA_SOURCE} | {CATEGORY} | eval crops {EVAL_CROPS} (headline {HEADLINE_CROP:.0%}) "
      f"| seeds {N_SEEDS_HEADLINE}/{N_SEEDS_OFFCROP} | tag={RUN_TAG}")

---
# A · Environment
> The PoinTr build cells are **her tested setup, kept essentially verbatim** (translated): CUDA arch
> auto-detect, `pointnet2_ops` as a pure-PyTorch shim (no compile), chamfer extension built from
> source. Only change: the fork is cloned from **our pinned copy** (fallback: hers). ~8 min on T4.

In [ ]:
# ============================================================
# Cell T1a — Dependencies + GPU check
# ------------------------------------------------------------
# torch 2.11 is built against numpy 2.x -> do NOT downgrade numpy (mixed install ->
# mtrand ABI error). Old pinned open3d/timm don't build on py3.12; current ones do.
# ============================================================
import subprocess, sys
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
      or "no GPU visible — Runtime > Change runtime type > GPU")
subprocess.run(sys.executable + " -m pip install -q easydict h5py matplotlib opencv-python pyyaml "
               "scipy tensorboardX tqdm transforms3d einops timm open3d gdown plotly wandb",
               shell=True, check=True)
import numpy as np, torch
print("deps OK | numpy", np.__version__, "(must be 2.x) | torch", torch.__version__)

In [ ]:
# ============================================================
# Cell T1b — PoinTr fork (pinned) + CUDA extensions + pointnet2_ops shim
# ------------------------------------------------------------
# Her hard-won build recipe, verbatim: (1) never hardcode the CUDA toolkit path or GPU
# arch (T4=7.5, V100=7.0, A100=8.0, L4=8.9 — wrong arch -> "no kernel image"); (2) do
# NOT compile pointnet2_ops (old CUDA repo won't build on torch 2.11+cu128) — inject a
# pure-PyTorch shim with the 6 functions PoinTr uses (tested against brute force);
# (3) build only the extensions PoinTr needs (emd is not one of them).
# ============================================================
import os, glob, subprocess, sys, types, torch

os.chdir("/content")
if not os.path.isdir("/content/PoinTr"):
    for url in [PINNED_FORK, FALLBACK_FORK]:
        if subprocess.run(f"git clone {url} /content/PoinTr", shell=True).returncode == 0:
            break
os.chdir("/content/PoinTr")
subprocess.run(f"git checkout {PIN_COMMIT}", shell=True)      # pinned: fork drift can't break us
print("PoinTr fork @", subprocess.run("git rev-parse --short HEAD", shell=True,
                                      capture_output=True, text=True).stdout.strip())

cuda_home = "/usr/local/cuda" if os.path.isdir("/usr/local/cuda") else sorted(glob.glob("/usr/local/cuda*"))[-1]
os.environ["CUDA_HOME"] = cuda_home
os.environ["PATH"] = f"{cuda_home}/bin:" + os.environ["PATH"]
cap = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cap[0]}.{cap[1]}"
print("CUDA_HOME", cuda_home, "| arch", os.environ["TORCH_CUDA_ARCH_LIST"])

# --- pointnet2_ops shim (pure torch; FPS loop is slow-ish but fine at ~8k points) ---
def furthest_point_sample(xyz, npoint):        # xyz (B,N,3) -> idx (B,npoint) int32
    B, N, _ = xyz.shape; dev = xyz.device
    idx = torch.zeros(B, npoint, dtype=torch.long, device=dev)
    dist = torch.full((B, N), 1e10, device=dev, dtype=xyz.dtype)
    far = torch.zeros(B, dtype=torch.long, device=dev); ar = torch.arange(B, device=dev)
    for i in range(npoint):
        idx[:, i] = far
        dist = torch.minimum(dist, ((xyz - xyz[ar, far].unsqueeze(1)) ** 2).sum(-1))
        far = torch.max(dist, dim=1).indices
    return idx.int()

def gather_operation(features, idx):           # (B,C,N),(B,S) -> (B,C,S)
    B, C, N = features.shape; idx = idx.long()
    return torch.gather(features, 2, idx.unsqueeze(1).expand(B, C, idx.shape[1])).contiguous()

def three_nn(query, ref):                      # (B,N,3),(B,M,3) -> dist (B,N,3), idx (B,N,3)
    d = torch.cdist(query, ref)
    dist, idx = torch.topk(d, 3, dim=-1, largest=False, sorted=True)
    return dist.contiguous(), idx.int().contiguous()

def three_interpolate(features, idx, weight):  # (B,C,M),(B,N,3),(B,N,3) -> (B,C,N)
    B, C, M = features.shape; N = idx.shape[1]; idx = idx.long()
    g = torch.gather(features, 2, idx.reshape(B,1,N*3).expand(B,C,N*3)).reshape(B,C,N,3)
    return (g * weight.unsqueeze(1)).sum(-1).contiguous()

def grouping_operation(features, idx):         # (B,C,N),(B,S,K) -> (B,C,S,K)
    B, C, N = features.shape; _, S, K = idx.shape; idx = idx.long()
    return torch.gather(features, 2, idx.reshape(B,1,S*K).expand(B,C,S*K)).reshape(B,C,S,K).contiguous()

def ball_query(radius, nsample, xyz, new_xyz): # -> idx (B,S,k)
    B, N, _ = xyz.shape; S = new_xyz.shape[1]
    d = torch.cdist(new_xyz, xyz)
    idx = torch.arange(N, device=xyz.device).view(1,1,N).expand(B,S,N).contiguous()
    idx[d > radius] = N
    idx = idx.sort(dim=-1).values[:, :, :nsample]
    first = idx[:, :, 0:1].clone(); first[first == N] = 0
    idx = torch.where(idx == N, first.expand(-1,-1,nsample), idx)
    return idx.int()

_u = types.ModuleType("pointnet2_ops.pointnet2_utils")
for _f in [furthest_point_sample, gather_operation, three_nn, three_interpolate,
           grouping_operation, ball_query]:
    setattr(_u, _f.__name__, _f)
_p = types.ModuleType("pointnet2_ops"); _p.pointnet2_utils = _u
sys.modules["pointnet2_ops"] = _p
sys.modules["pointnet2_ops.pointnet2_utils"] = _u
print("pointnet2_ops shim injected")

for ext in ["chamfer_dist", "gridding", "gridding_loss", "cubic_feature_sampling"]:
    r = subprocess.run("pip install -q --no-build-isolation .", shell=True,
                       cwd=f"/content/PoinTr/extensions/{ext}", capture_output=True, text=True)
    print(f"  {ext}:", "ok" if r.returncode == 0 else "FAILED\n" + r.stderr[-1500:])

In [ ]:
# ============================================================
# Cell T1c — Smoke test + our repo (tools/) + secrets
# ------------------------------------------------------------
# GPU smoke: fps + chamfer actually run. Then clone OUR repo for the tested modules and
# read tokens from Colab Secrets (HF_TOKEN, WANDB_API_KEY) — everything degrades
# gracefully to "off" so the science cells still run without them.
# ============================================================
import os, sys, subprocess
sys.path.insert(0, "/content/PoinTr"); os.chdir("/content/PoinTr")
import torch, numpy as np
from pointnet2_ops import pointnet2_utils
from extensions.chamfer_dist import ChamferDistanceL1
from models.PoinTr import PoinTr, fps
x = torch.rand(1, 1024, 3, device="cuda")
print("fps:", tuple(pointnet2_utils.furthest_point_sample(x, 128).shape),
      "| chamfer:", float(ChamferDistanceL1()(x, torch.rand(1, 512, 3, device="cuda"))))

REPO_DIR = "/content/point-cloud-completion-research"
if not os.path.isdir(REPO_DIR):
    subprocess.run(f"git clone --depth 1 {REPO_URL} {REPO_DIR}", shell=True, check=True)
# PoinTr ships its own REGULAR `tools` package (tools/__init__.py + runner/builder), and a
# regular package beats our namespace-style tools/ no matter the sys.path order — importing
# `tools.pc_repaint` would execute PoinTr's tools/__init__, which drags in emd (an extension
# we deliberately never build). So: put OUR tools dir itself on the path, import flat.
sys.path.insert(0, os.path.join(REPO_DIR, "tools"))
from pc_repaint import (Diffusion, PartColorDenoiser, train_color_ddpm, repaint_colors,
                        make_repaint_input, separate_colored, crop_xyz, deltaE,
                        nn_color, part_mean_colors, chamfer_l1, frame_scan, invert_frame,
                        apply_frame, save_ckpt, try_load_ckpt, pull_dir_from_hf,
                        load_done_keys, append_eval_row, read_rows, row_key,
                        summarize_rows, pairwise_diversity)
from pc_partlabels import (SEG_CLASSES, PART_NAMES_BY_CAT, load_h5_labeled_clouds,
                           build_labeled_set, load_labeled_npz, normalize_unit_sphere)
from pc_resilient import start_committer, hf_done_ids, anti_idle

HF_TOKEN = WANDB_KEY = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    print("HF token off:", type(e).__name__)
try:
    from google.colab import userdata
    WANDB_KEY = userdata.get("WANDB_API_KEY")
except Exception as e:
    print("wandb key off:", type(e).__name__)
DEV = "cuda"
print("tools ready | HF token:", bool(HF_TOKEN), "| wandb key:", bool(WANDB_KEY))
print("✅ environment READY")

---
# B · Data — materializes itself, no manual steps, no third-party dependency after first touch

* `"auto"` — ShapeNet-Part HDF5 benchmark (2048 pts + part labels; colors synthetic
  `PALETTE[part] + N(0, CNOISE)`). **Honest warning (hers, verified):** with oracle labels,
  part-mean is Bayes-optimal on this data — the `oracle ceiling` row is unbeatable; intra-part
  texture gains cannot show here. It's the machine-verification gate, not the result.
* `"labeled_s3"` — **real texture, built by us**: our HF airplane bundles (textured-mesh color
  sampling → EPFL dual-face cleanup → FPS 8192 — the same recipe as her `build_gt.py`) + ShapeNet-Part
  labels fused by NN transfer (`tools/pc_partlabels`, frame-QA'ed). Format-identical to her
  `labeled_s3` zip; if her zip ever lands next to this notebook it is ingested to HF as a
  continuity cross-check source.

All sources are **HF-first with mirror-on-first-touch**: h5 files and fused GT live in our public
dataset repo after the first run.

In [ ]:
# ============================================================
# Cell T2a — Materialize data sources (h5 mirror + fused labeled GT)
# ------------------------------------------------------------
# Idempotent: everything already on HF is skipped. The fusion driver is per-model atomic
# and QA-gated (mean NN distance; 48-frame search on v1/v2 mesh-frame mismatches) — a
# model with a bad label transfer is refused, not silently written (ADR-0008 rule:
# "done" = correct bytes on HF).
# ============================================================
import glob, json, os, shutil, urllib.request
from huggingface_hub import HfApi, hf_hub_download, list_repo_files
api = HfApi(token=HF_TOKEN)

H5_SRC = "larryshaw0079/ShapeNetPart"                     # public mirror, no token needed
ALL_SPLITS = ["train0","train1","train2","train3","train4","train5","val0","test0","test1"]
NEED = ALL_SPLITS if DATA_SOURCE == "labeled_s3" else AUTO_SPLITS
os.makedirs(LOCAL_H5, exist_ok=True)
have_hf = set(list_repo_files(HF_DATA_REPO, repo_type="dataset"))
for split in NEED:
    for fn in (f"{split}.h5", f"{split}_id2file.json"):
        dst = os.path.join(LOCAL_H5, fn)
        if os.path.exists(dst):
            continue
        if f"{H5_MIRROR}/{fn}" in have_hf:                # ours first
            p = hf_hub_download(HF_DATA_REPO, f"{H5_MIRROR}/{fn}", repo_type="dataset")
            shutil.copy(p, dst)
        else:                                             # third party once -> mirror to ours
            print("  fetching", fn, "from", H5_SRC)
            urllib.request.urlretrieve(f"https://huggingface.co/datasets/{H5_SRC}/resolve/main/{fn}", dst)
            if HF_TOKEN:
                api.upload_file(path_or_fileobj=dst, path_in_repo=f"{H5_MIRROR}/{fn}",
                                repo_id=HF_DATA_REPO, repo_type="dataset")
                print("    mirrored ->", f"{H5_MIRROR}/{fn}")
print("h5 ready:", sorted(os.listdir(LOCAL_H5))[:4], "...")

if DATA_SOURCE == "labeled_s3":
    # optional continuity source: her original zip, if someone dropped it next to us
    for z in ["/content/labeled_s3_02691156.zip", "/content/drive/MyDrive/labeled_s3_02691156.zip"]:
        if os.path.exists(z) and HF_TOKEN and "data_labeled/pelin_labeled_s3.zip" not in have_hf:
            api.upload_file(path_or_fileobj=z, path_in_repo="data_labeled/pelin_labeled_s3.zip",
                            repo_id=HF_DATA_REPO, repo_type="dataset")
            print("ingested her zip -> data_labeled/pelin_labeled_s3.zip (continuity cross-check)")

    anno = load_h5_labeled_clouds(LOCAL_H5, SYNSET)
    bundle_ids = {os.path.basename(f)[:-4] for f in have_hf
                  if f.startswith("data/") and f.endswith(".npz")}
    want = sorted(bundle_ids & set(anno))
    done_hf = hf_done_ids(HF_DATA_REPO, HF_TOKEN, path_in_repo=LABELED_PATH)
    os.makedirs(LOCAL_LABELED, exist_ok=True)
    for mid in done_hf:                                   # pull what exists (resume across VMs)
        dst = os.path.join(LOCAL_LABELED, f"{mid}.npz")
        if not os.path.exists(dst):
            shutil.copy(hf_hub_download(HF_DATA_REPO, f"{LABELED_PATH}/{mid}.npz",
                                        repo_type="dataset"), dst)
    print(f"labeled GT: {len(want)} candidates (bundles∩labels) | {len(done_hf)} already on HF")

    def load_bundle(mid):
        z = np.load(hf_hub_download(HF_DATA_REPO, f"data/{mid}.npz", repo_type="dataset"))
        return z["gt"]

    built, skipped, errs = build_labeled_set(want, load_bundle, anno, LOCAL_LABELED,
                                             SYNSET, done=done_hf)
    print(f"fusion: built {built}, skipped {skipped}, refused {len(errs)}")
    if built and HF_TOKEN:
        api.upload_folder(folder_path=LOCAL_LABELED, path_in_repo=LABELED_PATH,
                          repo_id=HF_DATA_REPO, repo_type="dataset",
                          allow_patterns=["*.npz"])
        print("labeled GT synced -> HF", LABELED_PATH)
    for mid, msg in errs[:5]:
        print("  refused:", mid, "|", msg)

In [ ]:
# ============================================================
# Cell T2b — Load DATA + split + the "is this data even textured?" verdict
# ------------------------------------------------------------
# Normalize BEFORE subsampling so the dense (PoinTr input) and sparse (color) clouds
# share one frame and the seed-only crop hits the SAME region at both resolutions
# (her fix). Split: sorted ids, last TEST_FRAC = test — deterministic across resumes.
# The intra-part ΔE printout is the part-mean rule's structural ceiling: if it is low,
# only the machine can be validated here, not the texture claim (her honest warning).
# ============================================================
import numpy as np

PALETTE = np.array([[.85,.20,.20],[.20,.65,.25],[.20,.35,.80],[.90,.75,.20],
                    [.60,.30,.75],[.25,.70,.70],[.90,.55,.20],[.5,.5,.5]])
P_GLOBAL = SEG_CLASSES[CATEGORY]
NUM_PARTS = len(P_GLOBAL)
PART_NAMES = (PART_NAMES_BY_CAT.get(CATEGORY) or [f"part{i}" for i in range(NUM_PARTS)])[:NUM_PARTS]

def _prep(xyz, rgb, part, j):
    xyz = normalize_unit_sphere(xyz)
    dense = np.ascontiguousarray(xyz).astype(np.float32)
    if len(xyz) != N_PTS:
        s = np.random.default_rng(j).choice(len(xyz), N_PTS, replace=len(xyz) < N_PTS)
        xyz, rgb, part = xyz[s], (rgb[s] if rgb is not None else None), part[s]
    return xyz.astype(np.float32), rgb, part.astype(np.int64), dense

DATA = []
if DATA_SOURCE == "auto":
    anno = load_h5_labeled_clouds(LOCAL_H5, SYNSET, splits=AUTO_SPLITS)
    rng_col = np.random.default_rng(0)
    for j, mid in enumerate(sorted(anno)):
        xyz, part = anno[mid]
        xyz, _, part, dense = _prep(xyz, None, part, j)
        rgb = np.clip(PALETTE[:NUM_PARTS][part] + rng_col.normal(0, CNOISE, (N_PTS, 3)),
                      0, 1).astype(np.float32)
        DATA.append(dict(model_id=mid, gt=np.concatenate([xyz, rgb], 1).astype(np.float32),
                         gt_part=part, dense=dense))
else:
    import glob, os
    files = sorted(glob.glob(os.path.join(LOCAL_LABELED, "*.npz")))
    assert files, "no labeled GT — run T2a (it builds them)"
    for j, f in enumerate(files):
        xyz, rgb, part = load_labeled_npz(f)
        xyz, rgb, part, dense = _prep(xyz, rgb, part, j)
        DATA.append(dict(model_id=os.path.basename(f)[:-4],
                         gt=np.concatenate([xyz, rgb.astype(np.float32)], 1).astype(np.float32),
                         gt_part=part, dense=dense))
if N_MODELS:
    DATA = DATA[:N_MODELS]

n_test = max(1, int(len(DATA) * TEST_FRAC))
TEST_IDX, TRAIN_IDX = list(range(len(DATA) - n_test, len(DATA))), list(range(len(DATA) - n_test))
print(f"DATA: {len(DATA)} models | {N_PTS} pts | {NUM_PARTS} parts {PART_NAMES} | "
      f"train {len(TRAIN_IDX)} / test {len(TEST_IDX)}")

intra = []
for d in DATA:
    g = d["gt"]
    for p in range(NUM_PARTS):
        m = d["gt_part"] == p
        if m.sum() > 5:
            intra.append(deltaE(g[m, 3:6], np.tile(g[m, 3:6].mean(0), (int(m.sum()), 1))).mean())
intra = float(np.mean(intra))
print("=" * 70)
print(f"  intra-part ΔE = {intra:.1f}   (= part-mean's structural CEILING)")
print("  textured — part-mean cannot go below this; RePaint can. That's the real contest."
      if intra > 12 else
      "  ⚠ flat colors — machine-verification stage only; texture gains cannot show here.")
print("=" * 70)

---
# C · Frozen PoinTr (geometry) — weights from our HF mirror, frame auto-scanned

PoinTr was trained on ShapeNet-55; our clouds arrive in a different axis convention. A wrong frame
makes PoinTr output an unrecognizable blob (this failure was hit before). T4 therefore scans **all
48 signed permutations** by Chamfer on a few train models — plus an input-density scan (PoinTr was
trained on 2048-6144-pt partials from 8192-pt clouds; a too-sparse partial degrades its kNN
grouping) — and prints a ✅/⚠️/⛔ verdict on whether completion actually beats the raw partial.

In [ ]:
# ============================================================
# Cell T3 — Frozen PoinTr + weights (HF mirror first, gdown once, mirror after)
# ------------------------------------------------------------
CKPT = "/content/PoinTr/ckpts/PoinTr_ShapeNet55.pth"
import os, shutil, subprocess
from easydict import EasyDict
from huggingface_hub import hf_hub_download
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 50e6:
    try:
        shutil.copy(hf_hub_download(HF_DATA_REPO, POINTR_CKPT_HF, repo_type="dataset"), CKPT)
        print("PoinTr weights <- our HF mirror")
    except Exception as e:
        print("HF mirror miss:", type(e).__name__, "-> gdown (once)")
        subprocess.run(f"gdown {POINTR_GDOWN_ID} -O {CKPT}", shell=True, check=True)
        if HF_TOKEN:
            from huggingface_hub import HfApi
            HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=CKPT, path_in_repo=POINTR_CKPT_HF,
                                              repo_id=HF_DATA_REPO, repo_type="dataset")
            print("mirrored ->", POINTR_CKPT_HF)
print("checkpoint MB:", round(os.path.getsize(CKPT) / 1e6, 1), "(should be >400)")

import torch
from models.PoinTr import PoinTr, fps
cfg = EasyDict(trans_dim=384, knn_layer=1, num_pred=6144, num_query=96)
geo = PoinTr(cfg)
sd = torch.load(CKPT, map_location="cpu")
base = sd.get("base_model", sd.get("model", sd))
base = {k.replace("module.", ""): v for k, v in base.items()}
mi, ui = geo.load_state_dict(base, strict=False)
assert len(mi) == 0, f"expected the original PoinTr, missing={mi[:4]}"
for p in geo.parameters():
    p.requires_grad_(False)
geo.eval().to(DEV)

AXIS_PERM, AXIS_SIGN = (2, 1, 0), (1, 1, 1)     # overwritten by the T4 scan

@torch.no_grad()
def complete_geometry(partial_xyz, perm=None, sign=None):
    perm = AXIS_PERM if perm is None else tuple(perm)
    sign = AXIS_SIGN if sign is None else tuple(sign)
    x = apply_frame(np.asarray(partial_xyz, np.float32)[:, :3], perm, sign)
    p = torch.from_numpy(x).float().unsqueeze(0).to(DEV)
    fine = geo(p)[1][0, :geo.num_pred].cpu().numpy()
    return invert_frame(fine, perm, sign)

def _fps_np(xyz, n):
    if len(xyz) <= n:
        return np.ascontiguousarray(xyz).astype(np.float32)
    t = torch.from_numpy(np.ascontiguousarray(xyz[:, :3])).float().unsqueeze(0).to(DEV)
    return fps(t, n)[0].cpu().numpy().astype(np.float32)

def pointr_input(d, crop, seed):
    src = crop_xyz(d["dense"], crop, seed) if USE_DENSE_FOR_POINTR else None
    if src is None:
        src = separate_colored(d["gt"], crop, seed)[0][:, :3]
    return _fps_np(src, POINTR_IN) if POINTR_IN and len(src) > POINTR_IN else src

def complete_of(d, crop, seed, perm=None, sign=None):
    return complete_geometry(pointr_input(d, crop, seed), perm, sign)

print(f"frozen PoinTr ready | num_pred={geo.num_pred}")

In [ ]:
# ============================================================
# Cell T4 — Frame scan (48 signed perms) + input-density scan + verdict
# ------------------------------------------------------------
# GT is available, so we measure instead of eyeballing. Reference = partial->GT Chamfer
# with NO completion: if the best completion isn't clearly better, the problem is not
# the frame (then suspect input density / point count — scanned right after).
# ============================================================
N_PROBE = 3
_probe = [DATA[i] for i in TRAIN_IDX[:N_PROBE]]
_ref = float(np.mean([chamfer_l1(separate_colored(d["gt"], HEADLINE_CROP, TRAIN_IDX[k])[0][:, :3],
                                 d["gt"][:, :3]) for k, d in enumerate(_probe)]))

_probes = [dict(partial=pointr_input(d, HEADLINE_CROP, TRAIN_IDX[k]), gt=d["gt"][:, :3])
           for k, d in enumerate(_probe)]
_scores = frame_scan(lambda pxyz, perm, sign: complete_geometry(pxyz, perm, sign), _probes)
_best, AXIS_PERM, AXIS_SIGN = _scores[0]
print(f"[1] FRAME scan ({len(_probes[0]['partial'])}-pt input)")
for _i, (_v, _p, _s) in enumerate(_scores[:5], 1):
    print(f"    {_i}  perm {list(_p)}  sign {list(_s)}  chamfer {_v:.4f}")
print(f"    worst: {_scores[-1][0]:.4f}  ->  AXIS_PERM={list(AXIS_PERM)} AXIS_SIGN={list(AXIS_SIGN)}")

_nd = len(_probe[0]["dense"])
_dres = []
for _kind, _n, _dense in ([("sparse", N_PTS // 2, False)] +
                          [("dense", _n, True) for _n in sorted({1024, 2048, 4096}) if _n <= _nd]):
    globals()["USE_DENSE_FOR_POINTR"], globals()["POINTR_IN"] = _dense, _n
    _c = [chamfer_l1(complete_of(d, HEADLINE_CROP, TRAIN_IDX[k]), d["gt"][:, :3])
          for k, d in enumerate(_probe)]
    _dres.append((float(np.mean(_c)), _kind, _n, _dense))
_dres.sort()
print("[2] INPUT-DENSITY scan")
for _v, _k, _n, _ in _dres:
    print(f"    {_k:<7} {_n:>5} pts   chamfer {_v:.4f}")
_bestd, _bk, POINTR_IN, USE_DENSE_FOR_POINTR = _dres[0]
print(f"    -> POINTR_IN={POINTR_IN} ({_bk} partial)")

print(f"[3] reference — partial->GT chamfer, no completion: {_ref:.4f} | best completion: {_bestd:.4f}")
if _bestd > _ref:
    print("    ⛔ completion WORSE than the raw partial -> PoinTr does not recognize this data.")
elif _bestd > 0.6 * _ref:
    print("    ⚠️ completion only slightly better — shapes may still be blobby; inspect T8.")
else:
    print("    ✅ completion clearly better — PoinTr recognizes the shapes.")

---
# D · PointNet part-seg + the two color DDPMs (part-conditional / part-blind)

All training is **config-signed + HF-resumable**: a checkpoint trained under any other config
(data source, category, sizes, split, architecture, epochs) is rejected and retrained — her
safety idea, moved from Drive to our HF runs directory. Training happens ONLY on the train split;
every table is measured on unseen test models. The DDPMs are trained **unconditionally — the
occlusion mask is never seen** (RePaint's core property).

In [ ]:
# ============================================================
# Cell T5 — PointNet part-seg (her STEP 2, verbatim) + config-signed ckpt
# ------------------------------------------------------------
# The part-conditioning stands on these labels; if they are bad, RePaint-part degrades
# (observed). The eval isolates that with the "oracle seg" rows — so we PRINT test
# accuracy + per-part coverage here and log them to the summary.
# ============================================================
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.spatial import cKDTree

CKPT_DIR = os.path.join(LOCAL_RUNS, "ckpts")
os.makedirs(CKPT_DIR, exist_ok=True)
pull_dir_from_hf(HF_DATA_REPO, RUNS_PATH, LOCAL_RUNS, token=HF_TOKEN)   # cross-VM resume
SCHED = start_committer(LOCAL_RUNS, HF_DATA_REPO, HF_TOKEN, every_min=10,
                        path_in_repo=RUNS_PATH) if (HF_SYNC_ON and HF_TOKEN) else None
if SCHED is None:
    print("HF sync off (no token) — results stay on this VM only")

DATA_SIG = dict(src=DATA_SOURCE, cat=CATEGORY, n_pts=N_PTS, cnoise=CNOISE,
                n_models=len(DATA), test_frac=TEST_FRAC, parts=NUM_PARTS,
                n_train=len(TRAIN_IDX), first_test=DATA[TEST_IDX[0]]["model_id"])

class _TNet(nn.Module):
    def __init__(s, k):
        super().__init__(); s.k = k
        s.mlp = nn.Sequential(nn.Conv1d(k,64,1),nn.BatchNorm1d(64),nn.ReLU(),
            nn.Conv1d(64,128,1),nn.BatchNorm1d(128),nn.ReLU(),
            nn.Conv1d(128,1024,1),nn.BatchNorm1d(1024),nn.ReLU())
        s.fc = nn.Sequential(nn.Linear(1024,512),nn.ReLU(),nn.Linear(512,256),nn.ReLU(),nn.Linear(256,k*k))
    def forward(s, x):
        B=x.size(0); f=s.mlp(x).max(-1)[0]
        return s.fc(f).view(B,s.k,s.k) + torch.eye(s.k,device=x.device).unsqueeze(0)

class PointNetPartSeg(nn.Module):
    def __init__(s, P):
        super().__init__(); s.itn=_TNet(3)
        s.mlp1=nn.Sequential(nn.Conv1d(3,64,1),nn.BatchNorm1d(64),nn.ReLU(),nn.Conv1d(64,128,1),nn.BatchNorm1d(128),nn.ReLU())
        s.fstn=_TNet(128)
        s.mlp2=nn.Sequential(nn.Conv1d(128,128,1),nn.BatchNorm1d(128),nn.ReLU(),nn.Conv1d(128,1024,1),nn.BatchNorm1d(1024),nn.ReLU())
        s.seg=nn.Sequential(nn.Conv1d(1152,512,1),nn.BatchNorm1d(512),nn.ReLU(),
            nn.Conv1d(512,256,1),nn.BatchNorm1d(256),nn.ReLU(),nn.Conv1d(256,P,1))
    def forward(s, x):
        x=x.transpose(1,2); x=torch.bmm(s.itn(x),x)
        f=s.mlp1(x); f=torch.bmm(s.fstn(f),f); pf=f
        g=s.mlp2(f).max(-1,keepdim=True)[0].expand(-1,-1,f.size(-1))
        return s.seg(torch.cat([pf,g],1)).transpose(1,2)

def train_partseg(model, loader, epochs=30, lr=1e-3, device=DEV):
    model.to(device).train(); opt=torch.optim.Adam(model.parameters(),lr); lf=nn.CrossEntropyLoss()
    for ep in range(epochs):
        cor=seen=0; tot=0.0
        for xyz,lab in loader:
            xyz,lab=xyz.to(device),lab.to(device); opt.zero_grad()
            lo=model(xyz); loss=lf(lo.reshape(-1,lo.size(-1)),lab.reshape(-1))
            loss.backward(); opt.step()
            tot+=loss.item()*xyz.size(0); cor+=(lo.argmax(-1)==lab).sum().item(); seen+=lab.numel()
        if ep%10==0 or ep==epochs-1: print(f"  ep{ep:3d} loss {tot/len(loader.dataset):.4f} acc {cor/seen*100:.1f}%")
    return model

@torch.no_grad()
def segment(model, xyz, device=DEV):
    model.eval(); x=np.asarray(xyz,np.float32)[:,:3]
    x=normalize_unit_sphere(x)
    return model(torch.from_numpy(x).float().unsqueeze(0).to(device))[0].argmax(-1).cpu().numpy()

class _GTPartDS(Dataset):
    def __init__(s, D): s.D = D
    def __len__(s): return len(s.D)
    def __getitem__(s, i):
        d = s.D[i]
        return torch.from_numpy(d["gt"][:, :3]).float(), torch.from_numpy(d["gt_part"]).long()

seg_model = PointNetPartSeg(NUM_PARTS).to(DEV)
seg_sig = dict(kind="pointnet_partseg", ep=SEG_EPOCHS, **DATA_SIG)
if try_load_ckpt(CKPT_DIR, "seg_model", seg_model, seg_sig, device=DEV):
    print("part-seg: LOADED from checkpoint (training skipped)")
else:
    train_partseg(seg_model, DataLoader(_GTPartDS([DATA[i] for i in TRAIN_IDX]),
                                        batch_size=16, shuffle=True), epochs=SEG_EPOCHS)
    save_ckpt(CKPT_DIR, "seg_model", seg_model, seg_sig)

def segment_oracle(d, xyz):
    # nearest-GT part label — isolates segmentation error: "is the coloring idea good"
    # and "is the segmenter good enough" are separate questions.
    x = normalize_unit_sphere(np.asarray(xyz, np.float32)[:, :3])
    g = normalize_unit_sphere(d["gt"][:, :3])
    _, j = cKDTree(g).query(x, k=1)
    return d["gt_part"][j]

_acc = [(segment(seg_model, DATA[i]["gt"][:, :3]) == DATA[i]["gt_part"]).mean() for i in TEST_IDX]
SEG_ACC = float(np.mean(_acc))
_cov = np.bincount(segment(seg_model, DATA[TEST_IDX[0]]["gt"][:, :3]), minlength=NUM_PARTS)
print(f"part-seg TEST accuracy: {SEG_ACC*100:.1f}% | coverage (1 test model): "
      f"{dict(zip(PART_NAMES, _cov.tolist()))}")
if (_cov == 0).any():
    print("  ⚠️ a part is NEVER predicted -> RePaint-part suffers; the 'oracle seg' rows show the true ceiling.")

In [ ]:
# ============================================================
# Cell T6 — Train the two color DDPMs (part-conditional + part-blind ablation)
# ------------------------------------------------------------
# UNCONDITIONAL training on full colored clouds — no mask, ever. One model then serves
# every occlusion ratio at eval time. W&B gets the eps-MSE curve per model.
# ============================================================
TRAIN_CLOUDS = [dict(xyz=DATA[i]["gt"][:, :3], rgb=DATA[i]["gt"][:, 3:6],
                     part=DATA[i]["gt_part"]) for i in TRAIN_IDX]
DIF = Diffusion(T=T_STEPS, device=DEV)
ARCH = dict(width=WIDTH, k=KNN, n_blocks=N_BLOCKS)

if WANDB_ON and WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)

def _train_one(name, part_cond):
    model = PartColorDenoiser(NUM_PARTS, part_cond=part_cond, **ARCH).to(DEV)
    sig = dict(kind="color_ddpm", T=T_STEPS, ep=DDPM_EPOCHS, part_cond=part_cond,
               **ARCH, **DATA_SIG)
    if try_load_ckpt(CKPT_DIR, name, model, sig, device=DEV):
        print(f"── {name}: LOADED from checkpoint (training skipped)")
        return model
    log_fn = None
    if WANDB_ON and WANDB_KEY:
        import wandb
        run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                         name=f"{RUN_TAG}_{name}", reinit=True,
                         config=dict(sig, n_train=len(TRAIN_CLOUDS)))
        log_fn = lambda ep, l: wandb.log({"eps_mse": l}, step=ep)
    print(f"── {name} training ({'part-conditional' if part_cond else 'part-blind ablation'})")
    train_color_ddpm(model, DIF, TRAIN_CLOUDS, epochs=DDPM_EPOCHS, bs=BATCH, lr=LR,
                     device=DEV, log=50, log_fn=log_fn)
    if log_fn:
        import wandb
        wandb.finish()
    save_ckpt(CKPT_DIR, name, model, sig)
    return model

ddpm_part = _train_one("ddpm_part", True)
ddpm_vanilla = _train_one("ddpm_vanilla", False)
if SCHED:
    SCHED.trigger()                      # checkpoints -> HF now, not in 10 minutes

---
# E · The formal eval — her six rows × the occlusion sweep, resumable per row

Metric: **mean ΔE(Lab) on the missing region**, unseen test models only. Rows per (table, crop):
NN-copy · part-mean · RePaint-vanilla (part-blind) · **RePaint-part** · RePaint-part (oracle seg) ·
oracle ceiling. Diffusion rows are stochastic → per-model seeds give **mean-of-seeds** (honesty row)
and **best-of-k** (does the prior cover the truth) + a **diversity** row (mean pairwise ΔE between
samples) — ADR-0009's two-row philosophy applied to color. `E2` = the motivating case: one part
fully occluded (dropped by GT label — dropping by the segmenter can silently degenerate, her trap).

Every measurement is one CSV row; done rows are skipped on re-run and the CSV syncs to HF every
few minutes — a runtime drop costs nothing but the row in flight.

In [ ]:
# ============================================================
# Cell T7 — Resilient eval driver (E1 sweep + E2 + diversity)  [hours; resumable]
# ------------------------------------------------------------
# Row key = (model, table, crop, variant, seed). Baselines are deterministic (seed 0).
# Crop seed = the model's DATA index (her convention -> parity with her tables at 50%).
# ============================================================
import time
anti_idle()
EVAL_CSV = os.path.join(LOCAL_RUNS, f"eval_rows_{DATA_SOURCE}.csv")
done = load_done_keys(EVAL_CSV)
print(f"resuming with {len(done)} rows already recorded")
test_ids = TEST_IDX if not EVAL_N else TEST_IDX[:EVAL_N]
DIFF_VARIANTS = [("RePaint-vanilla", "ddpm_vanilla", False),
                 ("RePaint-part", "ddpm_part", False),
                 ("RePaint-part (oracle seg)", "ddpm_part", True)]
MODELS = dict(ddpm_part=ddpm_part, ddpm_vanilla=ddpm_vanilla)

def put(row):
    if row_key(row) not in done:
        append_eval_row(EVAL_CSV, row)
        done.add(row_key(row))

t0 = time.time()
for crop in EVAL_CROPS:
    n_seeds = N_SEEDS_HEADLINE if abs(crop - HEADLINE_CROP) < 1e-9 else N_SEEDS_OFFCROP
    for n, i in enumerate(test_ids):
        d = DATA[i]
        need = [("E1", crop, v, s) for v, _, _ in DIFF_VARIANTS for s in range(n_seeds)]
        need += [("E1", crop, v, 0) for v in ("NN-copy", "part-mean", "oracle ceiling")]
        need += [("DIV", crop, v, 0) for v, _, _ in DIFF_VARIANTS]
        if all((d["model_id"], t, f"{c:.2f}", v, s) in done for t, c, v, s in need):
            continue
        gt, part_true = d["gt"], d["gt_part"]
        partial, miss = separate_colored(gt, crop, seed=i)
        comp = complete_of(d, crop, seed=i)
        _, gi = cKDTree(gt[:, :3]).query(comp[:, :3], k=1)
        true_rgb = gt[gi, 3:6]
        m = miss[gi]
        m = m if m.any() else np.ones(len(comp), bool)

        put(dict(model_id=d["model_id"], table="E1", crop=crop, variant="NN-copy", seed=0,
                 value=float(deltaE(nn_color(partial, comp)[m], true_rgb[m]).mean())))
        vl = segment(seg_model, partial[:, :3]); cl = segment(seg_model, comp)
        pm = part_mean_colors(partial[:, 3:6], vl, cl, NUM_PARTS)
        put(dict(model_id=d["model_id"], table="E1", crop=crop, variant="part-mean", seed=0,
                 value=float(deltaE(pm[m], true_rgb[m]).mean())))
        pf = np.stack([gt[part_true == p, 3:6].mean(0) if (part_true == p).any()
                       else gt[:, 3:6].mean(0) for p in range(NUM_PARTS)])
        put(dict(model_id=d["model_id"], table="E1", crop=crop, variant="oracle ceiling", seed=0,
                 value=float(deltaE(pf[part_true[gi]][m], true_rgb[m]).mean())))

        seg_fn = lambda xyz: segment(seg_model, xyz)
        ora_fn = lambda xyz: segment_oracle(d, xyz)
        inp = make_repaint_input(partial, comp, seg_fn, ora_fn, NUM_PARTS)
        inp_or = make_repaint_input(partial, comp, seg_fn, ora_fn, NUM_PARTS, use_oracle=True)
        for variant, mkey, use_or in DIFF_VARIANTS:
            ip = inp_or if use_or else inp
            samples = []
            for s in range(n_seeds):
                key = (d["model_id"], "E1", f"{crop:.2f}", variant, s)
                seed = int(crop * 100) * 100000 + 1000 * i + s
                rp = repaint_colors(MODELS[mkey], DIF, ip["xyz"], ip["part"], ip["known"],
                                    ip["rgb"], jump_length=JUMP_LEN, jump_n_sample=JUMP_N,
                                    adaptive_jumps=ADAPTIVE_JUMPS, seed=seed, device=DEV)
                samples.append(rp[ip["n_vis"]:])
                if key not in done:
                    put(dict(model_id=d["model_id"], table="E1", crop=crop, variant=variant,
                             seed=s, value=float(deltaE(samples[-1][m], true_rgb[m]).mean())))
            if len(samples) > 1:
                put(dict(model_id=d["model_id"], table="DIV", crop=crop, variant=variant, seed=0,
                         value=pairwise_diversity(samples, sel=m)))
        print(f"  crop {crop:.0%} [{n+1}/{len(test_ids)}] {d['model_id'][:10]} "
              f"({(time.time()-t0)/60:.1f} min)", end="\r")

# ---- E2: the fully-occluded part (headline crop only) ----
cnt = np.bincount(np.concatenate([DATA[i]["gt_part"] for i in test_ids]), minlength=NUM_PARTS)
TGT = int(np.argsort(-cnt)[1]) if NUM_PARTS > 1 else 0        # 2nd-largest part: harder
print(f"\nE2 target part: '{PART_NAMES[TGT]}'")
for n, i in enumerate(test_ids):
    d = DATA[i]
    gt, part_true = d["gt"], d["gt_part"]
    partial, miss = separate_colored(gt, HEADLINE_CROP, seed=i)
    comp = complete_of(d, HEADLINE_CROP, seed=i)
    _, gi = cKDTree(gt[:, :3]).query(comp[:, :3], k=1)
    true_rgb, gp = gt[gi, 3:6], part_true[gi]
    sel = gp == TGT
    if sel.sum() < 20:
        continue
    seg_fn = lambda xyz: segment(seg_model, xyz)
    ora_fn = lambda xyz: segment_oracle(d, xyz)
    inp = make_repaint_input(partial, comp, seg_fn, ora_fn, NUM_PARTS, drop_part=TGT)
    inp_or = make_repaint_input(partial, comp, seg_fn, ora_fn, NUM_PARTS, drop_part=TGT,
                                use_oracle=True)
    kvis = inp["known"][:inp["n_vis"]]
    vis_xyz, vis_rgb = inp["xyz"][:inp["n_vis"]][kvis], inp["rgb"][:inp["n_vis"]][kvis]
    vl = inp["part"][:inp["n_vis"]][kvis]
    _, j = cKDTree(vis_xyz).query(inp["xyz"][inp["n_vis"]:], k=1)
    put(dict(model_id=d["model_id"], table="E2", crop=HEADLINE_CROP, variant="NN-copy", seed=0,
             value=float(deltaE(vis_rgb[j][sel], true_rgb[sel]).mean())))
    pm = part_mean_colors(vis_rgb, vl, inp["part"][inp["n_vis"]:], NUM_PARTS)
    put(dict(model_id=d["model_id"], table="E2", crop=HEADLINE_CROP, variant="part-mean", seed=0,
             value=float(deltaE(pm[sel], true_rgb[sel]).mean())))
    for variant, mkey, use_or in DIFF_VARIANTS:
        ip = inp_or if use_or else inp
        for s in range(N_SEEDS_E2):
            key = (d["model_id"], "E2", f"{HEADLINE_CROP:.2f}", variant, s)
            if key in done:
                continue
            rp, diag = repaint_colors(MODELS[mkey], DIF, ip["xyz"], ip["part"], ip["known"],
                                      ip["rgb"], jump_length=JUMP_LEN, jump_n_sample=JUMP_N,
                                      adaptive_jumps=ADAPTIVE_JUMPS, seed=7000 + 1000 * i + s,
                                      device=DEV, return_diag=True)
            if use_or:
                assert TGT in diag["anchorless_parts"], "setup broken: target part not anchorless"
            put(dict(model_id=d["model_id"], table="E2", crop=HEADLINE_CROP, variant=variant,
                     seed=s, value=float(deltaE(rp[ip["n_vis"]:][sel], true_rgb[sel]).mean())))
    print(f"  E2 [{n+1}/{len(test_ids)}]", end="\r")

if SCHED:
    SCHED.trigger()

# ---- summary: honesty row (mean of seeds) + best-of-k + diversity ----
rows = read_rows(EVAL_CSV)
S = summarize_rows(rows)
ORDER = ["NN-copy", "part-mean", "RePaint-vanilla", "RePaint-part",
         "RePaint-part (oracle seg)", "oracle ceiling"]
for table in ("E1", "E2"):
    for crop in EVAL_CROPS:
        keys = [k for k in S if k[0] == table and abs(k[1] - crop) < 1e-9]
        if not keys:
            continue
        nm = max(S[k]["n_models"] for k in keys)
        print(f"\n{table} — ΔE(Lab) missing region | crop {crop:.0%} | {nm} unseen models | "
              f"part-seg {SEG_ACC*100:.1f}%")
        print("-" * 72)
        for v in ORDER:
            k = (table, crop, v)
            if k in S:
                best = f"   best-of-k {S[k]['best_mean']:6.2f}" if v.startswith("RePaint") else ""
                print(f"  {v:<26} {S[k]['mean']:7.3f} ± {S[k]['std']:5.3f}{best}")
        print("-" * 72)
div = [k for k in S if k[0] == "DIV"]
if div:
    print("\nDiversity (mean pairwise ΔE between samples, missing region):")
    for k in sorted(div, key=lambda x: (x[1], x[2])):
        print(f"  crop {k[1]:.0%} {k[2]:<26} {S[k]['mean']:6.2f}")
e1 = {(r["model_id"], r["variant"], r["seed"]): r["value"]
      for r in rows if r["table"] == "E1" and abs(r["crop"] - HEADLINE_CROP) < 1e-9}
mids = sorted({mid for mid, v, s in e1 if v == "NN-copy"})
wins = sum(np.mean([e1[m, "RePaint-part", s] for s in range(N_SEEDS_HEADLINE)
                    if (m, "RePaint-part", s) in e1]) < e1[m, "NN-copy", 0] for m in mids)
print(f"\nRePaint-part beats NN-copy on {wins}/{len(mids)} models @ {HEADLINE_CROP:.0%}")

if WANDB_ON and WANDB_KEY:
    import wandb
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=f"{RUN_TAG}_eval",
                     reinit=True, config=dict(DATA_SIG, crops=list(EVAL_CROPS),
                                              seg_acc=SEG_ACC, tag=RUN_TAG))
    tbl = wandb.Table(columns=["table", "crop", "variant", "mean", "std", "best_of_k", "n_models"])
    for (t, c, v), s in sorted(S.items()):
        tbl.add_data(t, c, v, s["mean"], s["std"], s["best_mean"], s["n_models"])
    wandb.log({"summary": tbl})
    wandb.finish()
print("\n✅ eval complete — rows on HF:", f"{RUNS_PATH}/eval_rows_{DATA_SOURCE}.csv")

In [ ]:
# ============================================================
# Cell T8 — Qualitative panel (first test model, headline crop) -> PNG for the report
# ------------------------------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

i = TEST_IDX[0]
d = DATA[i]
gt, partial = d["gt"], separate_colored(d["gt"], HEADLINE_CROP, seed=i)[0]
comp = complete_of(d, HEADLINE_CROP, seed=i)
seg_fn = lambda xyz: segment(seg_model, xyz)
ora_fn = lambda xyz: segment_oracle(d, xyz)
inp = make_repaint_input(partial, comp, seg_fn, ora_fn, NUM_PARTS)
rp_part = repaint_colors(ddpm_part, DIF, inp["xyz"], inp["part"], inp["known"], inp["rgb"],
                         jump_length=JUMP_LEN, jump_n_sample=JUMP_N, seed=0, device=DEV)
rp_van = repaint_colors(ddpm_vanilla, DIF, inp["xyz"], inp["part"], inp["known"], inp["rgb"],
                        jump_length=JUMP_LEN, jump_n_sample=JUMP_N, seed=0, device=DEV)
vl = segment(seg_model, partial[:, :3])
pm_rgb = part_mean_colors(partial[:, 3:6], vl, segment(seg_model, comp), NUM_PARTS)
nn_rgb = nn_color(partial, comp)

def full(rgb_comp):
    return np.vstack([partial[:, :3], comp]), np.vstack([partial[:, 3:6], rgb_comp])

panels = [("occluded (input)", (partial[:, :3], partial[:, 3:6])),
          ("NN-copy", full(nn_rgb)), ("part-mean", full(pm_rgb)),
          ("RePaint-vanilla", full(rp_van[inp["n_vis"]:])),
          ("RePaint-part", full(rp_part[inp["n_vis"]:])),
          ("GT", (gt[:, :3], gt[:, 3:6]))]
fig = plt.figure(figsize=(4 * len(panels), 4.2))
for c, (title, (x, r)) in enumerate(panels, 1):
    ax = fig.add_subplot(1, len(panels), c, projection="3d")
    ax.scatter(x[:, 0], x[:, 1], x[:, 2], c=np.clip(r, 0, 1), s=1.2)
    ax.set_title(title, fontsize=10); ax.set_axis_off()
    ax.set_box_aspect((np.ptp(x[:, 0]), np.ptp(x[:, 1]), np.ptp(x[:, 2])))
fig.tight_layout()
panel_path = os.path.join(LOCAL_RUNS, f"panel_{DATA_SOURCE}_{d['model_id'][:10]}.png")
fig.savefig(panel_path, dpi=140, bbox_inches="tight")
print("panel saved:", panel_path)
if SCHED:
    SCHED.trigger()
from IPython.display import Image, display
display(Image(panel_path))